# 04 – Analytics

Aggregation queries across all users and transactions.

Built-in methods:
- `spend_by_user(start, end)` — total per user
- `daily_spend(start, end)` — spend grouped by day
- `top_users(limit, start, end)` — highest spenders
- `aggregate_stats(start, end)` — summary stats

In [ ]:
from datetime import datetime, timedelta
from ducto.interface.postgres import PostgresStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PricingConfigV2, PlanDefinition,
    CreditMetadata,
)
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
import uuid, random
from datetime import timezone
print("✔ PostgresStore ready.")

### Seed sample data

Create 3 users with multiple reserve-then-deduct cycles over 7 days.

In [ ]:
users = [str(uuid.uuid4()) for _ in range(3)]
now = datetime.now(timezone.utc)

for u in users:
    store.add_credits(u, 100_000, type="adjustment")
    for day_offset in range(7):
        amount = random.randint(100, 2_000)
        res = store.reserve_credits(u, amount, operation_type="inference")
        store.deduct_credits(u, res.reservation_id, amount)

print(f"Seeded {3} users × 7 days of random transactions")

### Spend by user (last 30 days)

In [ ]:
from datetime import timedelta, timezone
end = datetime.now(timezone.utc)
start = end - timedelta(days=30)
rows = store.spend_by_user(start, end)
for r in rows:
    print(f"  {r.user_id[:8]}…  {r.total_spend:>7}  ({r.transaction_count} txns)")

### Daily spend

In [ ]:
rows = store.daily_spend(start, end)
print(f"{'Date':<12}  {'Spend':>7}  {'Txns':>5}")
for r in rows:
    print(f"{r.date:<12}  {r.total_spend:>7}  {r.transaction_count:>5}")

### Aggregate stats

In [ ]:
stats = store.aggregate_stats(start, end)
print(f"  Total consumed: {stats.total_credits_consumed}")
print(f"  Active users:   {stats.active_users}")
print(f"  Daily avg:      {stats.avg_daily_spend}")
print(f"  Top model:      {stats.top_model}")
print(f"  Top user:       {stats.top_user}")

In [ ]:
cleanup(pgdata)